# 使用 LlamaIndex 进行财务文档分析

在这个示例 notebook 中，我们将展示如何借助 [**LlamaIndex**](https://gpt-index.readthedocs.io/en/latest/) 框架，只用几行代码就对 [**10-K**](https://en.wikipedia.org/wiki/Form_10-K) 文档进行财务分析。

## Notebook 大纲
* [简介](#简介)
* [环境准备](#环境准备)
* [数据加载与索引](#数据加载与索引)
* [简单问答](#简单问答)
* [高级问答 - 对比分析](#高级问答---对比分析)


## 简介

### LLamaIndex
[LlamaIndex](https://gpt-index.readthedocs.io/en/latest/) 是一个面向 LLM 应用的数据框架。
只需几行代码即可上手，并在几分钟内构建一个检索增强生成（RAG）系统。
对于更高级的用户，LlamaIndex 提供了丰富的工具包，用于摄取和索引数据、检索与重排序模块，以及用于构建自定义查询引擎的可组合组件。

更多详情请参阅[完整文档](https://gpt-index.readthedocs.io/en/latest/)。

### 基于 10-K 文档的财务分析
财务分析师工作的一个关键部分，是从冗长的财务文档中提取信息并综合出洞见。
一个很好的例子是 10-K 表格——这是美国证券交易委员会（SEC）要求提交的年度报告，全面总结了公司的财务表现。
这些文档通常长达数百页，并包含特定领域的术语，外行人很难快速消化。


我们将展示 LlamaIndex 如何只需很少的代码，就能帮助财务分析师快速**跨多个文档**提取信息并综合洞见。

## 环境准备

首先，我们需要安装 llama-index 库

In [ ]:
!pip install llama-index pypdf

现在，导入本教程用到的所有模块

In [22]:
from langchain import OpenAI

from llama_index import SimpleDirectoryReader, ServiceContext, VectorStoreIndex
from llama_index import set_global_service_context
from llama_index.response.pprint_utils import pprint_response
from llama_index.tools import QueryEngineTool, ToolMetadata
from llama_index.query_engine import SubQuestionQueryEngine

开始之前，我们可以配置驱动 RAG 系统的 LLM 提供商和模型。  
这里我们选择 OpenAI 的 `gpt-3.5-turbo-instruct`。  

In [26]:
llm = OpenAI(temperature=0, model_name="gpt-3.5-turbo-instruct", max_tokens=-1)

我们构建一个 `ServiceContext` 并将其设为全局默认值，这样后续所有依赖 LLM 调用的操作都会使用我们在这里配置的模型。

In [28]:
service_context = ServiceContext.from_defaults(llm=llm)
set_global_service_context(service_context=service_context)

## 数据加载与索引

现在，我们加载并解析 2 个 PDF（一个是 Uber 2021 年的 10-K，另一个是 Lyft 2021 年的 10-K）。    
在底层，这些 PDF 会被转换为纯文本的 `Document` 对象，并按页拆分。  

> 注意：由于每份文档都超过 100 页，此操作可能需要运行一段时间。

In [30]:
lyft_docs = SimpleDirectoryReader(input_files=["../data/10k/lyft_2021.pdf"]).load_data()
uber_docs = SimpleDirectoryReader(input_files=["../data/10k/uber_2021.pdf"]).load_data()

In [31]:
print(f'Loaded lyft 10-K with {len(lyft_docs)} pages')
print(f'Loaded Uber 10-K with {len(uber_docs)} pages')

Loaded lyft 10-K with 238 pages
Loaded Uber 10-K with 307 pages


现在，我们可以基于已加载的文档构建一个（内存中的）`VectorStoreIndex`。  

> 注意：此操作可能需要运行一段时间，因为它会调用 OpenAI API 为文档块计算向量嵌入。

In [33]:
lyft_index = VectorStoreIndex.from_documents(lyft_docs)
uber_index = VectorStoreIndex.from_documents(uber_docs)

## 简单问答

现在我们可以对索引运行一些查询了！  
为此，我们先配置一个 `QueryEngine`，它只是记录了一组关于如何查询底层索引的配置。

对于 `VectorStoreIndex`，最常调整的配置是 `similarity_top_k`，它控制检索多少个文档块（我们称之为 `Node` 对象）作为回答问题的上下文。

In [34]:
lyft_engine = lyft_index.as_query_engine(similarity_top_k=3)

In [35]:
uber_engine = uber_index.as_query_engine(similarity_top_k=3)

来看看一些实际的查询吧！

In [79]:
response = await lyft_engine.aquery('What is the revenue of Lyft in 2021? Answer in millions with page reference')

In [80]:
print(response)


$3,208.3 million (page 63)


In [81]:
response = await uber_engine.aquery('What is the revenue of Uber in 2021? Answer in millions, with page reference')

In [82]:
print(response)


$17,455 (page 53)


## 高级问答 - 对比分析

对于更复杂的财务分析，通常需要参考多份文档。  

举个例子，我们来看看如何对 Lyft 和 Uber 的财务数据进行对比分析查询。  
为此，我们构建一个 `SubQuestionQueryEngine`，它会把复杂的对比分析查询拆解为更简单的子问题，并在由各自索引支撑的相应子查询引擎上执行。

In [83]:
query_engine_tools = [
    QueryEngineTool(
        query_engine=lyft_engine, 
        metadata=ToolMetadata(name='lyft_10k', description='Provides information about Lyft financials for year 2021')
    ),
    QueryEngineTool(
        query_engine=uber_engine, 
        metadata=ToolMetadata(name='uber_10k', description='Provides information about Uber financials for year 2021')
    ),
]

s_engine = SubQuestionQueryEngine.from_defaults(query_engine_tools=query_engine_tools)

来看看这些查询的实际效果！

In [84]:
response = await s_engine.aquery('Compare and contrast the customer segments and geographies that grew the fastest')

Generated 4 sub questions.
[uber_10k] Q: What customer segments grew the fastest for Uber
[uber_10k] A: in 2021?

The customer segments that grew the fastest for Uber in 2021 were its Mobility Drivers, Couriers, Riders, and Eaters. These segments experienced growth due to the continued stay-at-home order demand related to COVID-19, as well as Uber's introduction of its Uber One, Uber Pass, Eats Pass, and Rides Pass membership programs. Additionally, Uber's marketplace-centric advertising helped to connect merchants and brands with its platform network, further driving growth.
[uber_10k] Q: What geographies grew the fastest for Uber
[uber_10k] A: 
Based on the context information, it appears that Uber experienced the most growth in large metropolitan areas, such as Chicago, Miami, New York City, Sao Paulo, and London. Additionally, Uber experienced growth in suburban and rural areas, as well as in countries such as Argentina, Germany, Italy, Japan, South Korea, and Spain.
[lyft_10k] Q: 

In [85]:
print(response)


The customer segments that grew the fastest for Uber in 2021 were its Mobility Drivers, Couriers, Riders, and Eaters. These segments experienced growth due to the continued stay-at-home order demand related to COVID-19, as well as Uber's introduction of its Uber One, Uber Pass, Eats Pass, and Rides Pass membership programs. Additionally, Uber's marketplace-centric advertising helped to connect merchants and brands with its platform network, further driving growth. Uber experienced the most growth in large metropolitan areas, such as Chicago, Miami, New York City, Sao Paulo, and London. Additionally, Uber experienced growth in suburban and rural areas, as well as in countries such as Argentina, Germany, Italy, Japan, South Korea, and Spain.

The customer segments that grew the fastest for Lyft were ridesharing, light vehicles, and public transit. Ridesharing grew as Lyft was able to predict demand and proactively incentivize drivers to be available for rides in the right place at the r

In [86]:
response = await s_engine.aquery('Compare revenue growth of Uber and Lyft from 2020 to 2021')

Generated 2 sub questions.
[uber_10k] Q: What is the revenue growth of Uber from 2020 to 2021
[uber_10k] A: 
The revenue growth of Uber from 2020 to 2021 was 57%, or 54% on a constant currency basis.
[lyft_10k] Q: What is the revenue growth of Lyft from 2020 to 2021
[lyft_10k] A: 
The revenue growth of Lyft from 2020 to 2021 is 36%, increasing from $2,364,681 thousand to $3,208,323 thousand.


In [87]:
print(response)


The revenue growth of Uber from 2020 to 2021 was 57%, or 54% on a constant currency basis, while the revenue growth of Lyft from 2020 to 2021 was 36%. This means that Uber had a higher revenue growth than Lyft from 2020 to 2021.
